In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from tqdm import tqdm

In [4]:
import requests
from bs4 import BeautifulSoup

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

url = "https://www.flipkart.com/search?q=shoes"

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

links = []

for a in soup.select("a[href*='/p/']"):
    href = a.get("href")
    if href and "/p/" in href:
        full_link = "https://www.flipkart.com" + href.split("?")[0]
        links.append(full_link)

links = list(set(links))  # remove duplicates

print("Total links found:", len(links))
print(links[:5])

Total links found: 40
['https://www.flipkart.com/echor-casual-white-sneakers-men/p/itm0fa975f4d5cd5', 'https://www.flipkart.com/foot-trends-safety-shoes-steel-toes-industrial-construction-warehouse-use-men/p/itme7116d0ceeb1d', 'https://www.flipkart.com/urbanbox-sneakers-men/p/itmb3e2c6aa2ae94', 'https://www.flipkart.com/action-dotcom-drive-42-trendy-comfortable-durable-casual-slip-on-loafers-men/p/itm984b14cfeb63a', 'https://www.flipkart.com/franino-paris-amazing-design-women-s-ankle-length-block-heel-stylish-fashionable-boots-women/p/itm9565b3a4e797d']


In [5]:
import requests
from bs4 import BeautifulSoup
import time

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

base_url = "https://www.flipkart.com/search?q=shoes&page={}"

all_links = set()

for page in range(1, 30):   # ~30 pages ≈ 1200 links
    url = base_url.format(page)
    print("Scraping page:", page)

    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    for a in soup.select("a[href*='/p/']"):
        href = a.get("href")
        if href and "/p/" in href:
            full_link = "https://www.flipkart.com" + href.split("?")[0]
            all_links.add(full_link)

    time.sleep(2)  # avoid blocking

print("Total collected links:", len(all_links))

links = list(all_links)[:1000]  # keep first 1000
print("Final links:", len(links))

Scraping page: 1
Scraping page: 2
Scraping page: 3
Scraping page: 4
Scraping page: 5
Scraping page: 6
Scraping page: 7
Scraping page: 8
Scraping page: 9
Scraping page: 10
Scraping page: 11
Scraping page: 12
Scraping page: 13
Scraping page: 14
Scraping page: 15
Scraping page: 16
Scraping page: 17
Scraping page: 18
Scraping page: 19
Scraping page: 20
Scraping page: 21
Scraping page: 22
Scraping page: 23
Scraping page: 24
Scraping page: 25
Scraping page: 26
Scraping page: 27
Scraping page: 28
Scraping page: 29
Total collected links: 603
Final links: 603


Scrape ONE product including link

In [10]:
import requests
from bs4 import BeautifulSoup
import json

session = requests.Session()

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-IN,en;q=0.9"
}

# Step 1: open Flipkart homepage (important)
session.get("https://www.flipkart.com/", headers=headers)

# Step 2: open product page
url = "https://www.flipkart.com/foot-trends-safety-shoes-steel-toes-industrial-construction-warehouse-use-men/p/itme7116d0ceeb1d"
response = session.get(url, headers=headers)

soup = BeautifulSoup(response.text, "html.parser")

script = soup.find("script", {"id": "__NEXT_DATA__"})

if script:
    data_json = json.loads(script.string)
    product = data_json["props"]["pageProps"]["productData"]

    data = {
        "name": product.get("title"),
        "price": product.get("price", {}).get("finalPrice", {}).get("value"),
        "rating": product.get("rating", {}).get("average"),
        "rating_count": product.get("rating", {}).get("count"),
        "image": product.get("media", {}).get("images", [{}])[0].get("url"),
        "link": url
    }

    print(data)

else:
    print("JSON not found — still blocked")

JSON not found — still blocked


In [13]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

options = Options()
options.add_argument("--headless=new")      # headless browser
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

url = "https://www.flipkart.com/foot-trends-safety-shoes-steel-toes-industrial-construction-warehouse-use-men/p/itme7116d0ceeb1d"

driver.get(url)

print(driver.title)

WebDriverException: Message: unknown error: Chrome failed to start: exited normally.
  (unknown error: DevToolsActivePort file doesn't exist)
  (The process started from chrome location /snap/bin/chromium is no longer running, so ChromeDriver is assuming that Chrome has crashed.)
Stacktrace:
#0 0x5fe50c4924e3 <unknown>
#1 0x5fe50c1c1c76 <unknown>
#2 0x5fe50c1ead78 <unknown>
#3 0x5fe50c1e7029 <unknown>
#4 0x5fe50c225ccc <unknown>
#5 0x5fe50c22547f <unknown>
#6 0x5fe50c21cde3 <unknown>
#7 0x5fe50c1f22dd <unknown>
#8 0x5fe50c1f334e <unknown>
#9 0x5fe50c4523e4 <unknown>
#10 0x5fe50c4563d7 <unknown>
#11 0x5fe50c460b20 <unknown>
#12 0x5fe50c457023 <unknown>
#13 0x5fe50c4251aa <unknown>
#14 0x5fe50c47b6b8 <unknown>
#15 0x5fe50c47b847 <unknown>
#16 0x5fe50c48b243 <unknown>
#17 0x7a9d63094ac3 <unknown>


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))

url = "https://www.flipkart.com/foot-trends-safety-shoes-steel-toes-industrial-construction-warehouse-use-men/p/itme7116d0ceeb1d"

driver.get(url)

print(driver.title)